# Multimodal Deepfake Detection on a 970-Clip Balanced Subset

This notebook runs end-to-end training using a balanced 970-clip subset of the dataset, streams subprocess output live so cells do not look frozen, and packages the video, audio, and fusion artifacts for download.

## 1. Set Up Notebook Environment

In [ ]:
import json
import os
import random
import shutil
import subprocess
import time
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

# Kaggle-safe video profile: keep the backbone, but reduce loader pressure.
FREE_TIER_MODE = False
T4_OPTIMIZED_MODE = True
TARGET_SAMPLED_VIDEOS = 970

CONFIG = {
    "dataset_root": "/kaggle/input/fakebdteen",
    "working_dir": "/kaggle/working",
    "script_name": "train_mobilevit_video_deepfake.py",
    "output_dir": "/kaggle/working/outputs",
    "epochs": 10,
    "batch_size": 8,
    "lr": 1e-4,
    "weight_decay": 1e-2,
    "num_workers": 2,
    "num_frames": 8,
    "image_size": 224,
    "clip_seconds": 6,
    "train_ratio": 0.75,
    "val_ratio": 0.125,
    "max_train_samples": 0,
    "max_val_samples": 0,
    "max_test_samples": 0,
    "force_retrain": False,
}

def run_streaming_command(cmd, cwd=None, extra_env=None):
    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    env["PYTHONIOENCODING"] = "utf-8"
    if extra_env:
        env.update(extra_env)

    printable = " ".join(str(part) for part in cmd)
    print("Running:", printable, flush=True)
    started = time.perf_counter()
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=cwd,
        env=env,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    elapsed = time.perf_counter() - started
    print(f"Command finished in {elapsed:.1f}s with exit code {return_code}", flush=True)
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
print("FREE_TIER_MODE:", FREE_TIER_MODE)
print("T4_OPTIMIZED_MODE:", T4_OPTIMIZED_MODE)
print("TARGET_SAMPLED_VIDEOS:", TARGET_SAMPLED_VIDEOS)
print(json.dumps(CONFIG, indent=2))

In [ ]:
from collections import Counter, defaultdict
import shutil

RAW_DATASET_ROOT = Path(CONFIG["dataset_root"])
TARGET_SAMPLED_VIDEOS = 970
SAMPLED_DATASET_ROOT = Path(CONFIG["working_dir"]) / f"sampled_uniform_{TARGET_SAMPLED_VIDEOS}_videos"
SAMPLED_MANIFEST_PATH = SAMPLED_DATASET_ROOT / "sample_manifest.json"


def build_distributed_video_subset(raw_root: Path, target_total: int, seed: int):
    rng = random.Random(seed)
    folder_to_videos = defaultdict(list)

    for ext in ("*.mp4", "*.MP4"):
        for video_path in raw_root.rglob(ext):
            folder_to_videos[video_path.parent].append(video_path)

    if not folder_to_videos:
        raise RuntimeError(f"No videos found under {raw_root}")

    folders = sorted(folder_to_videos, key=lambda p: str(p).lower())
    base_quota = target_total // len(folders)
    quotas = {folder: min(len(folder_to_videos[folder]), base_quota) for folder in folders}
    remaining = target_total - sum(quotas.values())

    shuffled_folders = folders[:]
    rng.shuffle(shuffled_folders)
    while remaining > 0:
        progressed = False
        for folder in shuffled_folders:
            if remaining == 0:
                break
            if quotas[folder] < len(folder_to_videos[folder]):
                quotas[folder] += 1
                remaining -= 1
                progressed = True
        if not progressed:
            break

    if remaining > 0:
        raise RuntimeError(
            f"Could not reach target of {target_total} clips; only {target_total - remaining} are available after balancing."
        )

    selected_files = []
    folder_stats = []
    for folder in folders:
        videos = sorted(folder_to_videos[folder], key=lambda p: p.name.lower())
        chosen = rng.sample(videos, quotas[folder])
        chosen = sorted(chosen, key=lambda p: p.name.lower())
        selected_files.extend(chosen)
        folder_stats.append(
            {
                "folder": str(folder.relative_to(raw_root)),
                "available": len(videos),
                "selected": len(chosen),
            }
        )

    rng.shuffle(selected_files)
    return selected_files, folder_stats


def materialize_subset(selected_files, raw_root: Path, sampled_root: Path):
    if sampled_root.exists():
        shutil.rmtree(sampled_root)
    sampled_root.mkdir(parents=True, exist_ok=True)

    for src in selected_files:
        dst = sampled_root / src.relative_to(raw_root)
        dst.parent.mkdir(parents=True, exist_ok=True)
        try:
            os.symlink(src, dst)
        except OSError:
            shutil.copy2(src, dst)


selected_files, folder_stats = build_distributed_video_subset(RAW_DATASET_ROOT, TARGET_SAMPLED_VIDEOS, SEED)
materialize_subset(selected_files, RAW_DATASET_ROOT, SAMPLED_DATASET_ROOT)
CONFIG["dataset_root"] = str(SAMPLED_DATASET_ROOT)

sample_manifest = [
    {
        "relative_path": str(src.relative_to(RAW_DATASET_ROOT)),
        "folder": str(src.parent.relative_to(RAW_DATASET_ROOT)),
    }
    for src in selected_files
]
SAMPLED_MANIFEST_PATH.write_text(json.dumps(sample_manifest, indent=2))

folder_summary = Counter(item["folder"].split("/")[0] for item in folder_stats)
selected_counts = [item["selected"] for item in folder_stats]
print("Raw dataset root:", RAW_DATASET_ROOT)
print("Sampled dataset root:", SAMPLED_DATASET_ROOT)
print("Folders discovered:", len(folder_stats))
print("Videos selected:", len(selected_files))
print("Base quota per folder:", TARGET_SAMPLED_VIDEOS // len(folder_stats))
print("Per-top-level-folder counts:", dict(folder_summary))
print("Selected per-folder range:", min(selected_counts), "to", max(selected_counts))
print("Sample manifest saved to:", SAMPLED_MANIFEST_PATH)
print("Downstream training cells will now use the sampled subset only.")

In [ ]:
# Install dependencies (run once per Kaggle session)
%pip install -q transformers opencv-python-headless

## 2. Load Input Data

Locate the sampled dataset root, detect nested folder structure, and inspect discovered MP4 files.

In [ ]:
dataset_root = Path(CONFIG["dataset_root"])

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset root not found: {dataset_root}")

candidate_roots = [dataset_root] + [p for p in dataset_root.iterdir() if p.is_dir()]
resolved_root = None
for root in candidate_roots:
    has_mp4 = any(root.rglob("*.mp4")) or any(root.rglob("*.MP4"))
    if has_mp4:
        resolved_root = root
        break

if resolved_root is None:
    raise RuntimeError(f"No mp4 files found under {dataset_root}")

mp4_files = list(resolved_root.rglob("*.mp4")) + list(resolved_root.rglob("*.MP4"))
print("Resolved dataset root:", resolved_root)
print("Discovered videos:", len(mp4_files))
print("Sample files:")
for p in mp4_files[:5]:
    print(" -", p.name)

## 3. Validate and Clean Data

Parse filename schema and validate expected codes before training.

In [ ]:
from collections import Counter

valid_av = {"FF", "FR", "RF", "RR"}
valid_lang = {"0", "1"}
valid_gender = {"B", "G"}

meta_rows = []
invalid = []

for p in mp4_files:
    stem = p.stem
    if len(stem) < 6:
        invalid.append((p.name, "short_name"))
        continue
    av = stem[0:2].upper()
    lg = stem[2]
    gd = stem[3].upper()
    sid = stem[4:6]
    if av not in valid_av or lg not in valid_lang or gd not in valid_gender or not sid.isdigit():
        invalid.append((p.name, "schema_mismatch"))
        continue
    meta_rows.append((av, lg, gd, sid, p.name))

print("Valid files:", len(meta_rows))
print("Invalid files:", len(invalid))
if invalid[:5]:
    print("First invalid examples:", invalid[:5])

assert len(meta_rows) > 0, "No valid videos found after schema validation."

## 4. Exploratory Analysis with Plots

Show class distribution from filename metadata.

In [ ]:
import matplotlib.pyplot as plt

av_counts = Counter([r[0] for r in meta_rows])
lang_counts = Counter(["English" if r[1] == "0" else "Bangla" for r in meta_rows])
gender_counts = Counter(["Boy" if r[2] == "B" else "Girl" for r in meta_rows])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].bar(list(av_counts.keys()), list(av_counts.values()))
axes[0].set_title("AV Type Distribution")
axes[0].set_xlabel("Type")
axes[0].set_ylabel("Count")

axes[1].bar(list(lang_counts.keys()), list(lang_counts.values()))
axes[1].set_title("Language Distribution")

axes[2].bar(list(gender_counts.keys()), list(gender_counts.values()))
axes[2].set_title("Gender Distribution")

plt.tight_layout()
plt.show()

## 5. Build and Run Core Processing Logic

Ensure training script exists in Kaggle working directory and launch training.

In [ ]:
import shutil

working_script = Path(CONFIG["working_dir"]) / CONFIG["script_name"]

script_candidates = list(Path("/kaggle/input").rglob(CONFIG["script_name"]))
if script_candidates:
    latest_script = max(script_candidates, key=lambda p: p.stat().st_mtime)
    shutil.copy2(latest_script, working_script)
    print("Refreshed script from:", latest_script)
elif not working_script.exists():
    raise FileNotFoundError(
        f"{CONFIG['script_name']} not found in /kaggle/input. "
        f"Upload the script as a dataset file or add a notebook cell to write it first."
    )

print("Training script path:", working_script)
print("Script exists:", working_script.exists())

In [ ]:
video_done = (Path(CONFIG["output_dir"]) / "train_embeddings.pt").exists() and (Path(CONFIG["output_dir"]) / "metrics.pt").exists()
if video_done and not CONFIG.get("force_retrain", False):
    print("Video artifacts already found. Skipping training cell.")
else:
    video_cmd = [
        "python", "-u", "/kaggle/working/train_mobilevit_video_deepfake.py",
        "--dataset-root", CONFIG["dataset_root"],
        "--output-dir", CONFIG["output_dir"],
        "--epochs", str(CONFIG["epochs"]),
        "--batch-size", str(CONFIG["batch_size"]),
        "--lr", str(CONFIG["lr"]),
        "--weight-decay", str(CONFIG["weight_decay"]),
        "--num-workers", str(CONFIG["num_workers"]),
        "--num-frames", str(CONFIG["num_frames"]),
        "--image-size", str(CONFIG["image_size"]),
        "--clip-seconds", str(CONFIG["clip_seconds"]),
        "--train-ratio", str(CONFIG["train_ratio"]),
        "--val-ratio", str(CONFIG["val_ratio"]),
        "--max-train-samples", str(CONFIG["max_train_samples"]),
        "--max-val-samples", str(CONFIG["max_val_samples"]),
        "--max-test-samples", str(CONFIG["max_test_samples"]),
    ]
    run_streaming_command(video_cmd)

## 6. Evaluate Results with Metrics

Inspect generated outputs and report best validation accuracy.

In [ ]:
out_dir = Path(CONFIG["output_dir"])
print("Output dir exists:", out_dir.exists())
if out_dir.exists():
    for p in sorted(out_dir.glob("*")):
        print(" -", p.name)

metrics_path = out_dir / "metrics.pt"
if metrics_path.exists():
    metrics = torch.load(metrics_path, map_location="cpu")
    print("\nBest validation accuracy:", metrics.get("best_val_acc"))
    print("Split sizes:", {k: metrics.get(k) for k in ["num_train", "num_val", "num_test"]})
else:
    print("metrics.pt not found")

## 7. Save Outputs and Reusable Artifacts

Zip model checkpoint, embeddings, and metrics so they can be downloaded from Kaggle output.

In [ ]:
def zip_directory(source_dir: Path, zip_path: Path) -> Path:
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)
    if not source_dir.exists():
        raise FileNotFoundError(f"Source directory not found: {source_dir}")
    if zip_path.exists():
        zip_path.unlink()
    archive_base = zip_path.with_suffix("")
    shutil.make_archive(str(archive_base), "zip", root_dir=source_dir.parent, base_dir=source_dir.name)
    return zip_path


zip_path = Path(CONFIG["working_dir"]) / "mobilevit_outputs.zip"
if out_dir.exists():
    zip_directory(out_dir, zip_path)
    print("Created zip:", zip_path)
else:
    print("Output directory not found; run training cell first.")

---

# Wav2Vec2-Base Audio Deepfake Detection

This section trains a frozen Wav2Vec2-Base backbone with a trainable classification head on extracted audio from the same video dataset.

## Setup Audio Config

Free-tier mode is enabled by default to keep the pipeline within Kaggle session limits. Set FREE_TIER_MODE=False for full training.

In [ ]:
AUDIO_CONFIG = {
    "dataset_root": CONFIG["dataset_root"],
    "working_dir": CONFIG["working_dir"],
    "script_name": "train_wav2vec2_audio_deepfake.py",
    "output_dir": Path(CONFIG["working_dir"]) / "outputs_audio",
    "epochs": 12,
    "batch_size": 12,
    "lr": 1e-4,
    "weight_decay": 1e-2,
    "num_workers": 2,
    "ffmpeg_timeout": 45,
    "force_cpu": False,
    "train_ratio": 0.75,
    "val_ratio": 0.125,
    "max_train_samples": 0,
    "max_val_samples": 0,
    "max_test_samples": 0,
    "force_retrain": False,
}

print("Audio training config:")
print(json.dumps({k: str(v) if isinstance(v, Path) else v for k, v in AUDIO_CONFIG.items()}, indent=2))

In [ ]:
audio_working_script = Path(AUDIO_CONFIG["working_dir"]) / AUDIO_CONFIG["script_name"]

audio_script_candidates = list(Path("/kaggle/input").rglob(AUDIO_CONFIG["script_name"]))
if audio_script_candidates:
    latest_script = max(audio_script_candidates, key=lambda p: p.stat().st_mtime)
    shutil.copy2(latest_script, audio_working_script)
    print("Refreshed audio script from:", latest_script)
elif not audio_working_script.exists():
    raise FileNotFoundError(f"{AUDIO_CONFIG['script_name']} not found in /kaggle/input or /kaggle/working")

print("Audio training script path:", audio_working_script)
print("Script exists:", audio_working_script.exists())

In [ ]:
# Install ffmpeg for audio extraction (required for audio training)
!apt-get update && apt-get install -y ffmpeg > /dev/null 2>&1
print("ffmpeg installed.")

In [ ]:
audio_done = (Path(AUDIO_CONFIG["output_dir"]) / "train_embeddings.pt").exists() and (Path(AUDIO_CONFIG["output_dir"]) / "metrics.pt").exists()
if audio_done and not AUDIO_CONFIG.get("force_retrain", False):
    print("Audio artifacts already found. Skipping training cell.")
else:
    audio_cmd = [
        "python", "-u", "/kaggle/working/train_wav2vec2_audio_deepfake.py",
        "--dataset-root", str(AUDIO_CONFIG["dataset_root"]),
        "--output-dir", str(AUDIO_CONFIG["output_dir"]),
        "--epochs", str(AUDIO_CONFIG["epochs"]),
        "--batch-size", str(AUDIO_CONFIG["batch_size"]),
        "--lr", str(AUDIO_CONFIG["lr"]),
        "--weight-decay", str(AUDIO_CONFIG["weight_decay"]),
        "--num-workers", str(AUDIO_CONFIG["num_workers"]),
        "--ffmpeg-timeout", str(AUDIO_CONFIG["ffmpeg_timeout"]),
        "--train-ratio", str(AUDIO_CONFIG["train_ratio"]),
        "--val-ratio", str(AUDIO_CONFIG["val_ratio"]),
        "--max-train-samples", str(AUDIO_CONFIG["max_train_samples"]),
        "--max-val-samples", str(AUDIO_CONFIG["max_val_samples"]),
        "--max-test-samples", str(AUDIO_CONFIG["max_test_samples"]),
    ]
    if AUDIO_CONFIG.get("force_cpu", False):
        audio_cmd.append("--force-cpu")

    run_streaming_command(audio_cmd)

In [ ]:
audio_out_dir = Path(AUDIO_CONFIG["output_dir"])
print("Audio output dir exists:", audio_out_dir.exists())
if audio_out_dir.exists():
    for p in sorted(audio_out_dir.glob("*")):
        print(" -", p.name)

audio_metrics_path = audio_out_dir / "metrics.pt"
if audio_metrics_path.exists():
    audio_metrics = torch.load(audio_metrics_path, map_location="cpu")
    print("\nAudio Best validation accuracy:", audio_metrics.get("best_val_acc"))
    print("Audio Split sizes:", {k: audio_metrics.get(k) for k in ["num_train", "num_val", "num_test"]})
else:
    print("Audio metrics.pt not found")

## Late Fusion Setup

Load embeddings from both video and audio models for fusion experiments.

In [ ]:
video_embed_path = out_dir / "val_embeddings.pt"
audio_embed_path = audio_out_dir / "val_embeddings.pt"

if video_embed_path.exists() and audio_embed_path.exists():
    video_data = torch.load(video_embed_path, map_location="cpu")
    audio_data = torch.load(audio_embed_path, map_location="cpu")
    
    print("Video embeddings shape:", video_data["embeddings"].shape)
    print("Audio embeddings shape:", audio_data["embeddings"].shape)
    print("Video embedding dim (for fusion):", video_data["embeddings"].shape[1])
    print("Audio embedding dim (for fusion):", audio_data["embeddings"].shape[1])
    print("\nTotal embedding dims for fusion: {} + {} = {}".format(
        video_data["embeddings"].shape[1],
        audio_data["embeddings"].shape[1],
        video_data["embeddings"].shape[1] + audio_data["embeddings"].shape[1]
    ))
else:
    print("One or both embedding files not found.")
    if not video_embed_path.exists():
        print("Video:", video_embed_path)
    if not audio_embed_path.exists():
        print("Audio:", audio_embed_path)

In [ ]:
all_outputs_zip = Path(CONFIG["working_dir"]) / "all_deepfake_outputs.zip"
if out_dir.exists() or audio_out_dir.exists():
    !zip -r -q {all_outputs_zip} {out_dir} {audio_out_dir}
    print("Created comprehensive zip:", all_outputs_zip)
else:
    print("No output directories found.")

---

# Two-Head + 4-Class Multimodal Fusion

This section trains a multimodal model on frozen video/audio embeddings with:
- Video authenticity head (fake vs real)
- Audio authenticity head (fake vs real)
- Derived four-class prediction (FF, FR, RF, RR)

It includes frequency-aware filtering, cross-modality attention, progress bars, and checkpoint resume logic.

## Setup Fusion Config

In [ ]:
FUSION_CONFIG = {
    "script_name": "train_fusion_two_head_multimodal.py",
    "video_train_embed": str(out_dir / "train_embeddings.pt"),
    "audio_train_embed": str(audio_out_dir / "train_embeddings.pt"),
    "video_val_embed": str(out_dir / "val_embeddings.pt"),
    "audio_val_embed": str(audio_out_dir / "val_embeddings.pt"),
    "video_test_embed": str(out_dir / "test_embeddings.pt"),
    "audio_test_embed": str(audio_out_dir / "test_embeddings.pt"),
    "output_dir": str(Path(CONFIG["working_dir"]) / "outputs_fusion_two_head"),
    "epochs": 24,
    "batch_size": 192,
    "lr": 7e-4,
    "weight_decay": 1e-2,
    "num_workers": 2,
    "d_model": 256,
    "num_attn_heads": 8,
    "dropout": 0.2,
    "contrastive_weight": 0.05,
    "grad_clip_norm": 1.0,
    "early_stop_patience": 8,
    "resume": True,
    "force_retrain": False,
}

print("Fusion config:")
print(json.dumps(FUSION_CONFIG, indent=2))

In [ ]:
# Sanity check: verify embedding label semantics before two-head fusion training
video_train_path = Path(FUSION_CONFIG["video_train_embed"])
audio_train_path = Path(FUSION_CONFIG["audio_train_embed"])

if not video_train_path.exists() or not audio_train_path.exists():
    print("Embedding files are missing. Run video/audio embedding export first.")
else:
    v_data = torch.load(video_train_path, map_location="cpu")
    a_data = torch.load(audio_train_path, map_location="cpu")

    required_keys = ["labels", "paths", "av_type"]
    missing_v = [k for k in required_keys if k not in v_data]
    missing_a = [k for k in required_keys if k not in a_data]
    if missing_v or missing_a:
        print("Missing keys in embeddings.")
        print("Video missing:", missing_v)
        print("Audio missing:", missing_a)
    else:
        v_path_to_idx = {p: i for i, p in enumerate(v_data["paths"])}
        matched = []
        for a_idx, p in enumerate(a_data["paths"]):
            v_idx = v_path_to_idx.get(p)
            if v_idx is not None:
                matched.append((v_idx, a_idx))

        if not matched:
            print("No matched paths between video/audio embeddings.")
        else:
            v_idx = [x[0] for x in matched]
            a_idx = [x[1] for x in matched]

            v_labels = torch.as_tensor(v_data["labels"])[v_idx].long().numpy()
            a_labels = torch.as_tensor(a_data["labels"])[a_idx].long().numpy()
            av_types_v = [v_data["av_type"][i] for i in v_idx]
            av_types_a = [a_data["av_type"][i] for i in a_idx]

            if av_types_v != av_types_a:
                print("Warning: av_type mismatch between aligned video/audio embeddings.")

            expected_v = np.array([1 if av in {"FF", "FR"} else 0 for av in av_types_v], dtype=np.int64)
            expected_a = np.array([1 if av in {"FF", "RF"} else 0 for av in av_types_v], dtype=np.int64)

            v_ok = float((v_labels == expected_v).mean())
            a_ok = float((a_labels == expected_a).mean())

            print(f"Matched samples: {len(matched)}")
            print(f"Video label semantic match (fake video=1 for FF/FR): {v_ok:.4f}")
            print(f"Audio label semantic match (fake audio=1 for FF/RF): {a_ok:.4f}")
            print("Video label counts [real_video(0), fake_video(1)]:", np.bincount(v_labels, minlength=2).tolist())
            print("Audio label counts [real_audio(0), fake_audio(1)]:", np.bincount(a_labels, minlength=2).tolist())

            if v_ok < 0.999 or a_ok < 0.999:
                print("\nAction: regenerate embeddings by re-running video and audio training/export cells.")
            else:
                print("\nEmbedding semantics check PASSED. Safe to run two-head fusion training.")

In [ ]:
fusion_working_script = Path(CONFIG["working_dir"]) / FUSION_CONFIG["script_name"]

fusion_script_candidates = list(Path("/kaggle/input").rglob(FUSION_CONFIG["script_name"]))
if fusion_script_candidates:
    latest_script = max(fusion_script_candidates, key=lambda p: p.stat().st_mtime)
    shutil.copy2(latest_script, fusion_working_script)
    print("Refreshed fusion script from:", latest_script)
elif not fusion_working_script.exists():
    raise FileNotFoundError(f"{FUSION_CONFIG['script_name']} not found in /kaggle/input")

print("Fusion training script path:", fusion_working_script)
print("Script exists:", fusion_working_script.exists())

In [ ]:
fusion_done = (Path(FUSION_CONFIG["output_dir"]) / "test_metrics.pt").exists()
if fusion_done and not FUSION_CONFIG.get("force_retrain", False):
    print("Fusion artifacts already found. Skipping training cell.")
else:
    fusion_cmd = [
        "python", "-u", "/kaggle/working/train_fusion_two_head_multimodal.py",
        "--video-train-embed", FUSION_CONFIG["video_train_embed"],
        "--audio-train-embed", FUSION_CONFIG["audio_train_embed"],
        "--video-val-embed", FUSION_CONFIG["video_val_embed"],
        "--audio-val-embed", FUSION_CONFIG["audio_val_embed"],
        "--video-test-embed", FUSION_CONFIG["video_test_embed"],
        "--audio-test-embed", FUSION_CONFIG["audio_test_embed"],
        "--output-dir", FUSION_CONFIG["output_dir"],
        "--epochs", str(FUSION_CONFIG["epochs"]),
        "--batch-size", str(FUSION_CONFIG["batch_size"]),
        "--lr", str(FUSION_CONFIG["lr"]),
        "--weight-decay", str(FUSION_CONFIG["weight_decay"]),
        "--num-workers", str(FUSION_CONFIG["num_workers"]),
        "--d-model", str(FUSION_CONFIG["d_model"]),
        "--num-attn-heads", str(FUSION_CONFIG["num_attn_heads"]),
        "--dropout", str(FUSION_CONFIG["dropout"]),
        "--contrastive-weight", str(FUSION_CONFIG["contrastive_weight"]),
        "--grad-clip-norm", str(FUSION_CONFIG["grad_clip_norm"]),
        "--early-stop-patience", str(FUSION_CONFIG["early_stop_patience"]),
    ]
    if FUSION_CONFIG.get("resume", True):
        fusion_cmd.append("--resume")

    run_streaming_command(fusion_cmd)

## Evaluate Fusion Results

Display test metrics for:
- Video authenticity head accuracy
- Audio authenticity head accuracy
- Four-class (FF/FR/RF/RR) accuracy, macro F1, and confusion matrix

In [ ]:
fusion_out_dir = Path(FUSION_CONFIG["output_dir"])
fusion_metrics_path = fusion_out_dir / "test_metrics.pt"
fusion_json_path = fusion_out_dir / "test_metrics.json"

if fusion_metrics_path.exists():
    fusion_metrics = torch.load(fusion_metrics_path, map_location="cpu")
    print("=== Two-Head Multimodal Test Results ===")
    print(f"Video Head Accuracy: {fusion_metrics['video_head_accuracy']:.4f}")
    print(f"Audio Head Accuracy: {fusion_metrics['audio_head_accuracy']:.4f}")
    print(f"4-Class Accuracy: {fusion_metrics['four_class_accuracy']:.4f}")
    print(f"4-Class Macro F1: {fusion_metrics['four_class_f1_macro']:.4f}")
    print(f"Best Val Macro F1: {fusion_metrics.get('best_val_f1_macro', float('nan')):.4f}")
    print(f"Best Epoch: {fusion_metrics.get('best_epoch', -1)}")

    print("\n4-Class Labels Order:", fusion_metrics.get("class_names", ["FF", "FR", "RF", "RR"]))
    print("4-Class Confusion Matrix:")
    cm = np.array(fusion_metrics["four_class_confusion_matrix"])
    print(cm)

    if fusion_json_path.exists():
        print("\nJSON metrics path:", fusion_json_path)
else:
    print("Test metrics not found. Run fusion training cell first.")

In [ ]:
import zipfile


def zip_directory(source_dir: Path, zip_path: Path) -> Path:
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)
    if not source_dir.exists():
        raise FileNotFoundError(f"Source directory not found: {source_dir}")
    if zip_path.exists():
        zip_path.unlink()
    archive_base = zip_path.with_suffix("")
    shutil.make_archive(str(archive_base), "zip", root_dir=source_dir.parent, base_dir=source_dir.name)
    return zip_path


def zip_multiple_directories(zip_path: Path, directories) -> Path:
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for directory in directories:
            directory = Path(directory)
            if not directory.exists():
                continue
            for file_path in sorted(directory.rglob("*")):
                if file_path.is_file():
                    arcname = (Path(directory.name) / file_path.relative_to(directory)).as_posix()
                    archive.write(file_path, arcname=arcname)
    return zip_path


artifact_roots = [out_dir, audio_out_dir, fusion_out_dir]
manifest = []
for root in artifact_roots:
    if root.exists():
        files = sorted(root.glob("*"))
        print(f"Artifact folder: {root} ({len(files)} files)")
        for p in files:
            size_mb = round(p.stat().st_size / (1024 ** 2), 4)
            print(f" - {p.name} | {size_mb} MB")
            manifest.append({"folder": str(root), "file": p.name, "size_mb": size_mb})
    else:
        print(f"Artifact folder missing: {root}")

sample_manifest_path = SAMPLED_MANIFEST_PATH if SAMPLED_MANIFEST_PATH.exists() else None
manifest_path = Path(CONFIG["working_dir"]) / "artifact_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))

video_zip = zip_directory(out_dir, Path(CONFIG["working_dir"]) / "video_outputs.zip") if out_dir.exists() else None
audio_zip = zip_directory(audio_out_dir, Path(CONFIG["working_dir"]) / "audio_outputs.zip") if audio_out_dir.exists() else None
fusion_zip = zip_directory(fusion_out_dir, Path(CONFIG["working_dir"]) / "fusion_outputs.zip") if fusion_out_dir.exists() else None
bundle_items = [root for root in artifact_roots if root.exists()]
if sample_manifest_path is not None:
    bundle_items.append(sample_manifest_path)
combined_zip = Path(CONFIG["working_dir"]) / "all_deepfake_outputs.zip"
if bundle_items:
    if combined_zip.exists():
        combined_zip.unlink()
    with zipfile.ZipFile(combined_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for item in bundle_items:
            item = Path(item)
            if item.is_dir():
                for file_path in sorted(item.rglob("*")):
                    if file_path.is_file():
                        arcname = (Path(item.name) / file_path.relative_to(item)).as_posix()
                        archive.write(file_path, arcname=arcname)
            else:
                archive.write(item, arcname=(Path("metadata") / item.name).as_posix())

bundle_manifest = {
    "video_zip": str(video_zip) if video_zip else None,
    "audio_zip": str(audio_zip) if audio_zip else None,
    "fusion_zip": str(fusion_zip) if fusion_zip else None,
    "combined_zip": str(combined_zip) if combined_zip.exists() else None,
    "manifest_path": str(manifest_path),
    "sample_manifest_path": str(sample_manifest_path) if sample_manifest_path else None,
}
(Path(CONFIG["working_dir"]) / "download_index.json").write_text(json.dumps(bundle_manifest, indent=2))
print("Saved artifact manifest to:", manifest_path)
print("Download bundle index saved to:", Path(CONFIG["working_dir"]) / "download_index.json")
print("Available archives:")
for key, value in bundle_manifest.items():
    if value:
        print(f" - {key}: {value}")

## Single Video Inference (4-Class)

Run this section after training (or after downloading trained checkpoints). It predicts:
- Video head: real/fake video
- Audio head: real/fake audio
- Final 4-class label: FF, FR, RF, RR

In [ ]:
INFER_CONFIG = {
    "script_name": "infer_two_head_multimodal.py",
    "video_path": "/kaggle/working/input_video.mp4",  # set to your uploaded video path
    "fusion_checkpoint": str(Path(FUSION_CONFIG["output_dir"]) / "best_checkpoint.pt"),
    "output_json": "/kaggle/working/single_video_prediction.json",
    "num_frames": CONFIG.get("num_frames", 8),
    "image_size": CONFIG.get("image_size", 224),
    "clip_seconds": CONFIG.get("clip_seconds", 5),
    "audio_duration_sec": 10,
    "audio_sr": 16000,
    "ffmpeg_timeout": 45,
    "force_cpu": False,
}

print("Inference config:")
print(json.dumps(INFER_CONFIG, indent=2))

In [ ]:
infer_working_script = Path(CONFIG["working_dir"]) / INFER_CONFIG["script_name"]

infer_script_candidates = list(Path("/kaggle/input").rglob(INFER_CONFIG["script_name"]))
if infer_script_candidates:
    latest_script = max(infer_script_candidates, key=lambda p: p.stat().st_mtime)
    shutil.copy2(latest_script, infer_working_script)
    print("Refreshed inference script from:", latest_script)
elif not infer_working_script.exists():
    raise FileNotFoundError(f"{INFER_CONFIG['script_name']} not found in /kaggle/input")

print("Inference script path:", infer_working_script)
print("Script exists:", infer_working_script.exists())

In [ ]:
video_path = Path(INFER_CONFIG["video_path"])
checkpoint_path = Path(INFER_CONFIG["fusion_checkpoint"])

if not video_path.exists():
    print("Input video not found:", video_path)
    print("Upload or copy a video and set INFER_CONFIG['video_path'] correctly.")
elif not checkpoint_path.exists():
    print("Fusion checkpoint not found:", checkpoint_path)
    print("Run fusion training first or point to an existing checkpoint.")
else:
    infer_cmd = [
        "python", "-u", "/kaggle/working/infer_two_head_multimodal.py",
        "--video-path", str(video_path),
        "--fusion-checkpoint", str(checkpoint_path),
        "--output-json", INFER_CONFIG["output_json"],
        "--num-frames", str(INFER_CONFIG["num_frames"]),
        "--image-size", str(INFER_CONFIG["image_size"]),
        "--clip-seconds", str(INFER_CONFIG["clip_seconds"]),
        "--audio-duration-sec", str(INFER_CONFIG["audio_duration_sec"]),
        "--audio-sr", str(INFER_CONFIG["audio_sr"]),
        "--ffmpeg-timeout", str(INFER_CONFIG["ffmpeg_timeout"]),
    ]
    if INFER_CONFIG.get("force_cpu", False):
        infer_cmd.append("--force-cpu")

    run_streaming_command(infer_cmd)

    out_json = Path(INFER_CONFIG["output_json"])
    if out_json.exists():
        print("\nSaved prediction JSON:", out_json)
        print(out_json.read_text())

In [ ]:
from IPython.display import Markdown, display

report_lines = ["## Final Run Summary"]

video_metrics_path = out_dir / "metrics.pt"
audio_metrics_path = audio_out_dir / "metrics.pt"
fusion_metrics_path = fusion_out_dir / "test_metrics.pt"

if video_metrics_path.exists():
    video_metrics = torch.load(video_metrics_path, map_location="cpu")
    report_lines.append(f"- Video best validation accuracy: {video_metrics.get('best_val_acc', 'n/a')}")
    report_lines.append(
        f"- Video split sizes: train={video_metrics.get('num_train', 'n/a')}, val={video_metrics.get('num_val', 'n/a')}, test={video_metrics.get('num_test', 'n/a')}"
    )
else:
    report_lines.append("- Video metrics: not found")

if audio_metrics_path.exists():
    audio_metrics = torch.load(audio_metrics_path, map_location="cpu")
    report_lines.append(f"- Audio best validation accuracy: {audio_metrics.get('best_val_acc', 'n/a')}")
    report_lines.append(
        f"- Audio split sizes: train={audio_metrics.get('num_train', 'n/a')}, val={audio_metrics.get('num_val', 'n/a')}, test={audio_metrics.get('num_test', 'n/a')}"
    )
else:
    report_lines.append("- Audio metrics: not found")

if fusion_metrics_path.exists():
    fusion_metrics = torch.load(fusion_metrics_path, map_location="cpu")
    report_lines.append(f"- Fusion video-head accuracy: {fusion_metrics.get('video_head_accuracy', 'n/a')}")
    report_lines.append(f"- Fusion audio-head accuracy: {fusion_metrics.get('audio_head_accuracy', 'n/a')}")
    report_lines.append(f"- Fusion 4-class accuracy: {fusion_metrics.get('four_class_accuracy', 'n/a')}")
    report_lines.append(f"- Fusion 4-class macro F1: {fusion_metrics.get('four_class_f1_macro', 'n/a')}")
    report_lines.append(f"- Fusion best val macro F1: {fusion_metrics.get('best_val_f1_macro', 'n/a')}")
else:
    report_lines.append("- Fusion metrics: not found")

report_lines.append("## Download Paths")
for label, path in [
    ("Video outputs zip", Path(CONFIG["working_dir"]) / "video_outputs.zip"),
    ("Audio outputs zip", Path(CONFIG["working_dir"]) / "audio_outputs.zip"),
    ("Fusion outputs zip", Path(CONFIG["working_dir"]) / "fusion_outputs.zip"),
    ("Combined outputs zip", Path(CONFIG["working_dir"]) / "all_deepfake_outputs.zip"),
    ("Download index", Path(CONFIG["working_dir"]) / "download_index.json"),
]:
    if path.exists():
        report_lines.append(f"- {label}: {path}")
    else:
        report_lines.append(f"- {label}: not found")

display(Markdown("\n".join(report_lines)))